# Step N: Write back to DB
- This notebook creates a partitioned table

## 1. Setup Notebook Inputs
- path is where the data assets will be read in from
- Analysis file is the file created in step 1
- risk cluster file is the file coming from the risk and clustering notebooks
- table_prefix is the name of the table. The 

In [75]:
uat_connection = 'alex_uat'
path = '/project_data/data_asset/'
risk_cluster_file = 'cluster_risk_scores_04012020_06302020.csv'
table_prefix = 'RISK_SCORES'
#write_historical = False


## Import Libraries


In [76]:
import pandas as pd
import cx_Oracle
from sqlalchemy import create_engine, types
pd.set_option('display.max_columns', None)
from scipy import stats
import json
import re
import numpy as np

## Get credentials for connection

In [77]:
from project_lib import Project
project = Project.access()
uat_credentials = project.get_connection(name=uat_connection)

## Setup DB Connection

In [78]:
host = uat_credentials['host']
port = uat_credentials['port']
user = uat_credentials['username']
password = uat_credentials['password']
service_name = uat_credentials['service_name']

sid = cx_Oracle.makedsn(host = host, 
                        port = port, 
                        service_name = service_name)
 
cstr = 'oracle://{user}:{password}@{sid}'.format(
    user=user,
    password=password,
    sid=sid
)

engine =  create_engine(
    cstr,
    convert_unicode=False,
    pool_recycle=10,
    pool_size=50,
    echo=True
)

print(user)

u4j8175


## SQL functions

In [79]:
# This will check if a table exists then delete it if so
def drop_table_if_exists(table_name):
    qry = """
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || \'{tbl}\';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
                 RAISE;
              END IF;
        END;    
    """.format(tbl = table_name)
    res = engine.execute(qry)
    return('ran')
 
# This will check if a table exists in a user's schema    
def check_if_exists(table_name):
    qry = 'SELECT table_name FROM user_tables'
    res = pd.read_sql(qry, engine)
    out = any(res.table_name.isin([table_name]))
    return(out)    

## Step 0. Write historical assets back to DB (will skip in future)
- Get the assets from the file folder
- Filter the assets to the files we want to write
- Write data back to database
- setup _latest and _Previous files

In [80]:
# all_assets = project.get_assets()
# asset_names = [asset['name'] for asset in all_assets]
# assets_to_write = ['cluster_risk_scores' in i for i in asset_names]
# idx = [i for i, val in enumerate(assets_to_write) if val]
# final_list = [asset_names[i] for i in idx]
# final_list = list(np.setdiff1d(final_list,final_list[-1]))

# print(final_list)

In [81]:
# latest = final_list[len(final_list)-1]
# previous = final_list[len(final_list)-2]
# latest_name = 'RISK_LATEST'
# previous_name = 'RISK_PREVIOUS'

# if write_historical:
#     for asset in final_list:
#         print(asset)
#         tempdf = pd.read_csv('/project_data/data_asset/' + asset)
#         date_asset = tempdf['Min Dispense Date'].min()
#         tempdf['Min Dispense Date'] = pd.to_datetime(tempdf['Min Dispense Date'])
#         tempdf['Max Dispense Date'] = pd.to_datetime(tempdf['Max Dispense Date'])
#         data_to_write = tempdf
#         table_name = 'RISK_' + re.sub("-","_",date_asset)
#         print(table_name)

#         drop_table_if_exists(table_name)
#         data_to_write.to_sql(table_name, engine, if_exists='replace', index=False, dtype={"prscrb_prov_loc_id": types.VARCHAR(256)})
        
#         if asset == latest:
#             drop_table_if_exists(latest_name)
#             data_to_write.to_sql(latest_name, engine, if_exists='replace', index=False, dtype={"prscrb_prov_loc_id": types.VARCHAR(256)})
            
#         if asset == previous:
#             drop_table_if_exists(previous_name)
#             data_to_write.to_sql(previous_name, engine, if_exists='replace', index=False, dtype={"prscrb_prov_loc_id": types.VARCHAR(256)})            

## Step 1. Read in final risk score file

In [82]:
current_quarter_df = pd.read_csv(path + risk_cluster_file)
current_quarter_df['Min Dispense Date'] = pd.to_datetime(current_quarter_df['Min Dispense Date'])
current_quarter_df['Max Dispense Date'] = pd.to_datetime(current_quarter_df['Max Dispense Date'])
current_quarter_df['Bin'] = current_quarter_df['Bin'].astype(str)

current_quarter_df.head()

,Member Count,Count Script,Sum Dispensed Quantity,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,Total Days,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30,Risk_Score,Bin,Cluster_label
0,32,80,4119.0,2020-06-29,2020-04-01,105691,90,2.5,128.718750,0.888889,45.766667,0.060000,0.03125,5,0.0,0.0,0.0,0.1125,0.32,0,0,0.0,0.0,6.439075,0.0,0.0,0.0,0.03125,0.0,0.21875,0.0,0.03125,9.079689,Low Risk,1
1,4,4,266.0,2020-06-19,2020-04-29,105693,52,1.0,66.500000,0.076923,5.115385,0.061224,0.00000,0,0.0,0.0,0.0,0.2500,0.00,0,0,0.0,0.0,1.758067,0.0,0.0,0.0,0.00000,0.0,0.00000,0.0,0.00000,0.803871,Low Risk,2
2,1,1,30.0,2020-04-30,2020-04-30,105699,1,1.0,30.000000,1.000000,30.000000,0.000000,0.00000,0,0.0,0.0,0.0,0.0000,0.00,0,0,0.0,0.0,13.271845,0.0,0.0,0.0,0.00000,0.0,0.00000,0.0,0.00000,2.276211,Low Risk,2
3,5,9,215.0,2020-06-10,2020-05-18,105700,24,1.8,43.000000,0.375000,8.958333,0.108696,0.00000,3,0.0,0.0,0.0,1.0000,0.00,0,0,0.0,0.0,6.675857,0.0,0.0,0.0,0.00000,0.0,0.40000,0.0,0.00000,4.025246,Low Risk,2
4,3,6,340.0,2020-06-29,2020-04-04,105707,87,2.0,113.333333,0.068966,3.908046,0.032258,0.00000,0,0.0,0.0,0.0,0.0000,0.00,0,0,0.0,0.0,17.707574,0.0,0.0,0.0,0.00000,0.0,0.00000,0.0,0.00000,1.900472,Low Risk,2


## Step 2. Write current quarter risk scores back to the DB

In [83]:
data_to_write = current_quarter_df
date_asset = str(current_quarter_df['Max Dispense Date'].max())[0:10]
table_name = table_prefix + '_' + re.sub("-","_",date_asset)
print(table_name)

drop_table_if_exists(table_name)
data_to_write.to_sql(table_name, engine, if_exists='replace', index=False, dtype={"PRSCRB_PROV_LOC_ID": types.VARCHAR(256),
                                                                                  "Bin": types.VARCHAR(256)})

RISK_SCORES_2020_06_30
2020-10-29 18:15:03,156 INFO sqlalchemy.engine.base.Engine SELECT USER FROM DUAL
2020-10-29 18:15:03,158 INFO sqlalchemy.engine.base.Engine {}
2020-10-29 18:15:03,203 INFO sqlalchemy.engine.base.Engine SELECT CAST('test plain returns' AS VARCHAR(60 CHAR)) AS anon_1 FROM DUAL
2020-10-29 18:15:03,204 INFO sqlalchemy.engine.base.Engine {}
2020-10-29 18:15:03,249 INFO sqlalchemy.engine.base.Engine SELECT CAST('test unicode returns' AS NVARCHAR2(60)) AS anon_1 FROM DUAL
2020-10-29 18:15:03,250 INFO sqlalchemy.engine.base.Engine {}
2020-10-29 18:15:03,334 INFO sqlalchemy.engine.base.Engine select value from nls_session_parameters where parameter = 'NLS_NUMERIC_CHARACTERS'
2020-10-29 18:15:03,335 INFO sqlalchemy.engine.base.Engine {}
2020-10-29 18:15:03,379 INFO sqlalchemy.engine.base.Engine 
        BEGIN
           EXECUTE IMMEDIATE 'DROP TABLE ' || 'RISK_SCORES_2020_06_30';
        EXCEPTION
           WHEN OTHERS THEN
              IF SQLCODE != -942 THEN
          

2020-10-29 18:15:04,854 INFO sqlalchemy.engine.base.Engine COMMIT
2020-10-29 18:15:04,945 INFO sqlalchemy.engine.base.Engine SELECT table_name FROM all_tables WHERE nvl(tablespace_name, 'no tablespace') NOT IN ('SYSTEM', 'SYSAUX') AND OWNER = :owner AND IOT_NAME IS NULL AND DURATION IS NULL
2020-10-29 18:15:04,946 INFO sqlalchemy.engine.base.Engine {'owner': 'U4J8175'}


/opt/conda/envs/Python-3.6-WMLCE/lib/python3.6/site-packages/pandas/io/sql.py:1191: UserWarning: The provided table name 'RISK_SCORES_2020_06_30' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  warnings.warn(msg, UserWarning)


## Step 3. Setup _Previous and _Latest table

In [84]:
# #1 remove _Previous table
# print(check_if_exists('RISK_LATEST'))
# drop_table_if_exists('RISK_PREVIOUS')
# engine.execute('create table RISK_PREVIOUS as select * from RISK_LATEST')

# #2 add latest

# drop_table_if_exists('RISK_LATEST')
# data_to_write.to_sql('RISK_LATEST', engine, if_exists='replace', index=False, dtype={"PRSCRB_PROV_LOC_ID": types.VARCHAR(256),
#                                                                                   "Bin": types.VARCHAR(256)})

## Step 4. Read in latest table

In [85]:
# latest = pd.read_sql('select * from RISK_LATEST where rownum < 10', engine)
# latest.head()

## Step 5. Read in previous table

In [86]:
# latest = pd.read_sql('select * from RISK_PREVIOUS where rownum < 10', engine)
# latest.head()

In [87]:
# pd.read_sql('select * from user_tables', engine)